# 📊 Notebook 10 — Observability and Governance Evidence

This notebook inspects the governance traces, routing audit events, and telemetry generated by the scenario runs.

## What we observe
| Signal | Source | What to look for |
|--------|--------|------------------|
| Agent call traces | Application Insights | Which agents were invoked per request |
| Routing decisions | OTel logs (`agt.audit`) | Intent classification, agent selection, governance gates |
| Disclaimer events | Orchestrator logs | Requests blocked by disclaimer gate |
| Confidence scores | Agent responses | Requests refused due to low confidence |
| Token usage | APIM / Cosmos DB | Model consumption per agent via APIM |

## Prerequisites
- Notebooks 7, 8, 9 executed (generated traces)
- Application Insights resource available in the hub

In [ ]:
import sys, json, pathlib, subprocess

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing shared/utils.py and workshop/product-finder")

repo_root = find_repo_root(pathlib.Path.cwd())
sys.path.insert(0, str(repo_root / "shared"))
import utils  # type: ignore

def run(cmd: str, ok: str = "", fail: str = ""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

hub_rg = azd_get("AZURE_RESOURCE_GROUP")
utils.print_info(f"Hub resource group: {hub_rg}")

### 1️⃣ Find Application Insights workspace

In [ ]:
ai_out = run(
    f'az resource list -g {hub_rg} --resource-type microsoft.insights/components -o json',
    'AppInsights query OK', 'AppInsights query failed'
)
if not ai_out.success or not ai_out.json_data:
    raise RuntimeError('No Application Insights resource found in hub RG.')

ai_name = ai_out.json_data[0]['name']
utils.print_ok(f'Application Insights: {ai_name}')

ai_detail = run(
    f'az monitor app-insights component show --app {ai_name} -g {hub_rg} -o json',
    '', ''
)
ai_app_id = ai_detail.json_data.get('appId', '') if ai_detail.success and ai_detail.json_data else ''
utils.print_info(f'App ID: {ai_app_id}')

### 2️⃣ Query agent traces from Application Insights

This query retrieves the last 50 requests to Product Finder agents, showing which agents were called and their durations.

In [ ]:
TRACES_QUERY = '''
requests
| where cloud_RoleName has "pf-"
| where timestamp > ago(2h)
| project timestamp, cloud_RoleName, name, duration, resultCode, customDimensions
| order by timestamp desc
| take 50
'''.strip()

traces_out = run(
    f'az monitor app-insights query --app {ai_name} -g {hub_rg} '
    f'--analytics-query "{TRACES_QUERY}" -o json',
    'Traces query OK', 'Traces query failed'
)
if traces_out.success and traces_out.json_data:
    rows = traces_out.json_data.get('tables', [{}])[0].get('rows', [])
    if rows:
        print(f'\nFound {len(rows)} agent request traces:\n')
        print(f'  {"Timestamp":<25}  {"Agent":<35}  {"Duration(ms)":<14}  Status')
        print(f'  {"-"*25}  {"-"*35}  {"-"*14}  ------')
        for row in rows[:20]:
            print(f'  {str(row[0])[:24]:<25}  {str(row[1]):<35}  {str(row[3])[:12]:<14}  {row[4]}')
    else:
        utils.print_warning('No agent traces found in last 2h. Run the scenario notebooks first.')
else:
    utils.print_warning('Could not query Application Insights — check permissions.')

### 3️⃣ Query governance audit events (`agt.audit`)

The orchestrator and governed agents emit structured audit events via OpenTelemetry.
This query retrieves governance decision records.

In [ ]:
AUDIT_QUERY = '''
traces
| where cloud_RoleName has "pf-"
| where message has "governance" or message has "routing" or message has "disclaimer" or message has "DENY" or message has "ALLOW"
| where timestamp > ago(2h)
| project timestamp, cloud_RoleName, message, customDimensions
| order by timestamp desc
| take 30
'''.strip()

audit_out = run(
    f'az monitor app-insights query --app {ai_name} -g {hub_rg} '
    f'--analytics-query "{AUDIT_QUERY}" -o json',
    'Audit query OK', 'Audit query failed'
)
if audit_out.success and audit_out.json_data:
    rows = audit_out.json_data.get('tables', [{}])[0].get('rows', [])
    if rows:
        print(f'\nGovernance audit events ({len(rows)} found):\n')
        for row in rows[:15]:
            icon = '✅' if 'ALLOW' in str(row[2]) else ('❌' if 'DENY' in str(row[2]) else '📋')
            print(f'  {icon}  [{str(row[0])[:19]}]  {str(row[1]):<30}  {str(row[2])[:80]}')
    else:
        utils.print_warning('No audit events found. Ensure ENABLE_INSTRUMENTATION=true on agents.')
else:
    utils.print_warning('Could not query audit logs.')

### 4️⃣ Query Cosmos DB for usage records

All LLM calls route through APIM and are logged to Cosmos DB via the Citadel usage ingestion pipeline.

In [ ]:
cosmos_out = run(
    f'az resource list -g {hub_rg} --resource-type Microsoft.DocumentDB/databaseAccounts -o json',
    '', ''
)
if not cosmos_out.success or not cosmos_out.json_data:
    utils.print_warning('Cosmos DB not found in hub RG — skipping usage query.')
else:
    cosmos_name = cosmos_out.json_data[0]['name']
    utils.print_ok(f'Cosmos DB account: {cosmos_name}')
    print()
    print('  To view Product Finder usage records, open the Azure Portal and run:')
    print('  SELECT * FROM c WHERE c.model LIKE "%Product-Finder%" ORDER BY c._ts DESC OFFSET 0 LIMIT 20')
    print(f'  in the Cosmos DB account: {cosmos_name} → usage database')

print()
print('=' * 65)
print('  GOVERNANCE EVIDENCE CHECKLIST')
print('=' * 65)
checks = [
    'Agent traces visible per agent role in Application Insights',
    'Routing decisions recorded in audit logs (agt.audit.event)',
    'Disclaimer gate events present for compatibility queries',
    'Multi-agent bundle attribution tracked per response',
    'All LLM calls routed through APIM (Cosmos DB usage records)',
]
for c in checks:
    print(f'  ☐  {c}')
print()
utils.print_ok('✅ Observability notebook COMPLETE. Product Finder workshop DONE!')